# Step 2b: Train COOLANT (Phased, Experimental)

Phased pre-training with image-caption pairs:
- Phase 1: CLIP pre-training (self-supervised)
- Phase 2: Similarity pre-training (matched/unmatched)
- Phase 3: Detection training (balanced pairs)
- Phase 4: Joint fine-tuning (0.1x LR)

Input: `./processed/coolant_{train,dev,test}.h5`
Output: `../../training/checkpoints_coolant_phased/`

In [1]:
import sys, os, math, random, warnings, datetime
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, accuracy_score

warnings.filterwarnings("ignore")

project_root = Path().absolute().parent.parent
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "src"))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

SAVE_DIR = project_root / "training" / "checkpoints_coolant_phased"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


In [2]:
# Load data and model (same as 2a)
from src.processing.coolant import create_coolant_dataloaders
from src.processing.coolant.training_utils import make_coolant_pairs, make_detection_batch, soft_cross_entropy
from src.models.resnet_coolant import ResNetCOOLANT, patch_encoding, patch_clip_projection, patch_cnn_with_dropout

HDF5_DIR = Path("./processed")
BATCH_SIZE = 32
IMAGE_DIM = 2048
TEXT_EMBED = 768

loaders, datasets = create_coolant_dataloaders(
    str(HDF5_DIR / "coolant_train.h5"), str(HDF5_DIR / "coolant_dev.h5"), str(HDF5_DIR / "coolant_test.h5"),
    batch_size=BATCH_SIZE,
)

CONFIG = {"shared_dim": 128, "sim_dim": 64, "clip_embed_dim": 64, "feature_dim": 96, "h_dim": 64,
    "lr": 1e-3, "lr_clip": 1e-3, "weight_decay": 1e-5, "label_smoothing": 0.1, "grad_clip": 1.0,
    "max_epochs_phase1": 10, "max_epochs_phase2": 10, "max_epochs_phase3": 20, "max_epochs_phase4": 10,
    "patience": 5, "dropout": 0.3}

model = ResNetCOOLANT(CONFIG)
patch_encoding(model.similarity_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.encoding, image_dim=IMAGE_DIM)
patch_encoding(model.detection_module.ambiguity_module.encoding, image_dim=IMAGE_DIM)
patch_clip_projection(model.clip_module, target_dim=IMAGE_DIM, is_image=True)
patch_clip_projection(model.clip_module, target_dim=TEXT_EMBED, is_image=False)
patch_cnn_with_dropout(model.similarity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
patch_cnn_with_dropout(model.detection_module.ambiguity_module.encoding.shared_text_encoding, TEXT_EMBED, CONFIG["dropout"])
model = model.to(DEVICE)

loss_cos = nn.CosineEmbeddingLoss(margin=0.2)
loss_ce = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
loss_kl = nn.KLDivLoss(reduction="batchmean")
print("Model and losses ready")

CoolantPairDataset: 3537 pairs from coolant_train.h5
CoolantPairDataset: 1054 pairs from coolant_dev.h5
CoolantPairDataset: 876 pairs from coolant_test.h5

COOLANT DataLoaders created:
  Train: 111 batches (3537 pairs)
  Dev:   33 batches (1054 pairs)
  Test:  28 batches (876 pairs)
Model and losses ready


In [4]:
# Phase 1: CLIP pre-training (self-supervised)
print("=" * 50)
print("PHASE 1: CLIP Pre-training")
print("=" * 50)

for name, param in model.named_parameters():
    param.requires_grad = "clip_module" in name

optim_clip = torch.optim.AdamW(model.clip_module.parameters(), lr=CONFIG["lr_clip"], weight_decay=5e-4)
best_clip_loss = float("inf")

for epoch in range(1, CONFIG["max_epochs_phase1"] + 1):
    model.train()
    epoch_loss, n = 0.0, 0
    for caption, image, _ in tqdm(loaders["train"], desc=f"P1 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        bs = caption.size(0)
        if bs < 4: continue
        cap_pooled = caption.mean(dim=2)
        ie, te = model.clip_module(image, cap_pooled)
        logits = ie @ te.T * math.exp(0.07)
        ids = torch.arange(bs, device=DEVICE)
        lc = (loss_ce(logits, ids) + loss_ce(logits.T, ids)) / 2
        optim_clip.zero_grad(); lc.backward()
        torch.nn.utils.clip_grad_norm_(model.clip_module.parameters(), CONFIG["grad_clip"])
        optim_clip.step()
        epoch_loss += lc.item() * bs; n += bs
    print(f"  Epoch {epoch}: loss={epoch_loss/n:.4f}")
    if epoch_loss/n < best_clip_loss:
        best_clip_loss = epoch_loss/n
        torch.save(model.clip_module.state_dict(), SAVE_DIR / "clip_phase1.pth")
print(f"Phase 1 done. Best loss: {best_clip_loss:.4f}")

PHASE 1: CLIP Pre-training


P1 Ep1: 100%|██████████| 111/111 [02:01<00:00,  1.10s/it]


  Epoch 1: loss=2.7014


P1 Ep2:  28%|██▊       | 31/111 [00:33<01:26,  1.08s/it]


KeyboardInterrupt: 

In [ ]:
# Phase 2: Similarity pre-training
print("\n" + "=" * 50)
print("PHASE 2: Similarity Pre-training")
print("=" * 50)

for param in model.clip_module.parameters(): param.requires_grad = False
for param in model.similarity_module.parameters(): param.requires_grad = True
for param in model.detection_module.parameters(): param.requires_grad = False

optim_sim = torch.optim.Adam(model.similarity_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
best_sim_loss = float("inf")

for epoch in range(1, CONFIG["max_epochs_phase2"] + 1):
    model.train()
    epoch_loss, n_batches = 0.0, 0
    for caption, image, _ in tqdm(loaders["train"], desc=f"P2 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        if caption.size(0) < 4: continue
        cap_a, img_m, img_u = make_coolant_pairs(caption, image)
        ta_m, ia_m, _ = model.similarity_module(cap_a, img_m)
        ta_u, ia_u, _ = model.similarity_module(cap_a, img_u)
        t_cat = torch.cat([ta_m, ta_u])
        i_cat = torch.cat([ia_m, ia_u])
        y_cos = torch.cat([torch.ones(ta_m.size(0), device=DEVICE), -torch.ones(ta_u.size(0), device=DEVICE)])
        ls = loss_cos(t_cat, i_cat, y_cos)
        optim_sim.zero_grad(); ls.backward()
        torch.nn.utils.clip_grad_norm_(model.similarity_module.parameters(), CONFIG["grad_clip"])
        optim_sim.step()
        epoch_loss += ls.item(); n_batches += 1
    avg = epoch_loss / n_batches
    print(f"  Epoch {epoch}: loss={avg:.4f}")
    if avg < best_sim_loss:
        best_sim_loss = avg
        torch.save(model.similarity_module.state_dict(), SAVE_DIR / "sim_phase2.pth")
print(f"Phase 2 done. Best loss: {best_sim_loss:.4f}")

In [ ]:
# Phase 3: Detection training
print("\n" + "=" * 50)
print("PHASE 3: Detection Training")
print("=" * 50)

for param in model.clip_module.parameters(): param.requires_grad = False
for param in model.similarity_module.parameters(): param.requires_grad = False
for param in model.detection_module.parameters(): param.requires_grad = True

optim_det = torch.optim.Adam(model.detection_module.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
best_val_f1 = 0.0
patience = 0

for epoch in range(1, CONFIG["max_epochs_phase3"] + 1):
    model.train()
    all_y, all_p = [], []
    for caption, image, _ in tqdm(loaders["train"], desc=f"P3 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        if caption.size(0) < 4: continue
        det_cap, det_img, det_labels = make_detection_batch(caption, image)
        cap_pooled = det_cap.mean(dim=2)
        with torch.no_grad():
            ie, te = model.clip_module(det_img, cap_pooled)
        det_out, attn, skl = model.detection_module(det_cap, det_img, te, ie)
        ld = loss_ce(det_out, det_labels) + 0.5 * loss_kl(F.log_softmax(attn, 1), F.softmax(skl, 1))
        optim_det.zero_grad(); ld.backward()
        torch.nn.utils.clip_grad_norm_(model.detection_module.parameters(), CONFIG["grad_clip"])
        optim_det.step()
        all_y.extend(det_labels.cpu().numpy()); all_p.extend(det_out.argmax(1).cpu().numpy())

    train_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)

    # Validate
    model.eval()
    val_y, val_p = [], []
    with torch.no_grad():
        for caption, image, _ in loaders["dev"]:
            caption, image = caption.to(DEVICE), image.to(DEVICE)
            if caption.size(0) < 4: continue
            det_cap, det_img, det_labels = make_detection_batch(caption, image)
            cap_pooled = det_cap.mean(dim=2)
            ie, te = model.clip_module(det_img, cap_pooled)
            det_out, _, _ = model.detection_module(det_cap, det_img, te, ie)
            val_y.extend(det_labels.cpu().numpy()); val_p.extend(det_out.argmax(1).cpu().numpy())

    val_f1 = f1_score(val_y, val_p, average="macro", zero_division=0)
    cm = confusion_matrix(val_y, val_p)
    print(f"  Epoch {epoch}: train_F1={train_f1:.4f} val_F1={val_f1:.4f} CM={cm.tolist()}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1; patience = 0
        torch.save(model.detection_module.state_dict(), SAVE_DIR / "det_phase3.pth")
        print(f"  >> Best F1={best_val_f1:.4f}")
    else:
        patience += 1
        if patience >= CONFIG["patience"]: print(f"Early stopping"); break

print(f"Phase 3 done. Best F1: {best_val_f1:.4f}")

In [ ]:
# Phase 4: Joint fine-tuning
print("\n" + "=" * 50)
print("PHASE 4: Joint Fine-tuning")
print("=" * 50)

for param in model.parameters(): param.requires_grad = True
optim_joint = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"] * 0.1, weight_decay=CONFIG["weight_decay"])
best_joint_f1 = 0.0
patience = 0

for epoch in range(1, CONFIG["max_epochs_phase4"] + 1):
    model.train()
    all_y, all_p = [], []
    for caption, image, _ in tqdm(loaders["train"], desc=f"P4 Ep{epoch}"):
        caption, image = caption.to(DEVICE), image.to(DEVICE)
        bs = caption.size(0)
        if bs < 4: continue

        # Similarity
        cap_a, img_m, img_u = make_coolant_pairs(caption, image)
        ta_m, ia_m, _ = model.similarity_module(cap_a, img_m)
        ta_u, ia_u, _ = model.similarity_module(cap_a, img_u)
        ls = loss_cos(torch.cat([ta_m, ta_u]), torch.cat([ia_m, ia_u]),
            torch.cat([torch.ones(ta_m.size(0), device=DEVICE), -torch.ones(ta_u.size(0), device=DEVICE)]))

        # CLIP
        cap_pooled = caption.mean(dim=2)
        ie, te = model.clip_module(image, cap_pooled)
        logits = ie @ te.T * math.exp(0.07)
        ids = torch.arange(bs, device=DEVICE)
        l_clip = (loss_ce(logits, ids) + loss_ce(logits.T, ids)) / 2

        # Detection
        det_cap, det_img, det_labels = make_detection_batch(caption, image)
        det_cap_pooled = det_cap.mean(dim=2)
        ie2, te2 = model.clip_module(det_img, det_cap_pooled)
        det_out, attn, skl = model.detection_module(det_cap, det_img, te2, ie2)
        ld = loss_ce(det_out, det_labels) + 0.5 * loss_kl(F.log_softmax(attn, 1), F.softmax(skl, 1))

        total_loss = ls + l_clip + ld
        optim_joint.zero_grad(); total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
        optim_joint.step()
        all_y.extend(det_labels.cpu().numpy()); all_p.extend(det_out.argmax(1).cpu().numpy())

    train_f1 = f1_score(all_y, all_p, average="macro", zero_division=0)
    model.eval()
    val_y, val_p = [], []
    with torch.no_grad():
        for caption, image, _ in loaders["dev"]:
            caption, image = caption.to(DEVICE), image.to(DEVICE)
            if caption.size(0) < 4: continue
            det_cap, det_img, det_labels = make_detection_batch(caption, image)
            cap_pooled = det_cap.mean(dim=2)
            ie, te = model.clip_module(det_img, cap_pooled)
            det_out, _, _ = model.detection_module(det_cap, det_img, te, ie)
            val_y.extend(det_labels.cpu().numpy()); val_p.extend(det_out.argmax(1).cpu().numpy())

    val_f1 = f1_score(val_y, val_p, average="macro", zero_division=0)
    cm = confusion_matrix(val_y, val_p)
    print(f"  Epoch {epoch}: train_F1={train_f1:.4f} val_F1={val_f1:.4f} CM={cm.tolist()}")

    if val_f1 > best_joint_f1:
        best_joint_f1 = val_f1; patience = 0
        torch.save(model.state_dict(), SAVE_DIR / "full_model_phase4.pth")
        print(f"  >> Best F1={best_joint_f1:.4f}")
    else:
        patience += 1
        if patience >= CONFIG["patience"]: print(f"Early stopping"); break

print(f"\nAll phases complete.")
print(f"  Phase 1 CLIP loss: {best_clip_loss:.4f}")
print(f"  Phase 2 Sim loss:  {best_sim_loss:.4f}")
print(f"  Phase 3 Det F1:    {best_val_f1:.4f}")
print(f"  Phase 4 Joint F1:  {best_joint_f1:.4f}")